# Multi-Provider Conversational Agent with MiniMax Support

## Overview
This tutorial demonstrates how to build a conversational agent that can work with **multiple LLM providers** through a single, unified interface. In addition to OpenAI, we show how to use [MiniMax](https://www.minimaxi.com/) (M2.7 / M2.5 / M2.5-highspeed) as an alternative LLM backend.

## Motivation
Most GenAI agent tutorials are hard-wired to a single provider (typically OpenAI). In production you often need to:
- **Switch providers** without rewriting agent code.
- **Reduce costs** by routing certain workloads to cheaper models.
- **Improve resilience** by falling back to a secondary provider when the primary is unavailable.

MiniMax M2.7 is the latest high-capability model with a **204K token context window** and an OpenAI-compatible API, making it a practical alternative.

## Key Components
1. **`utils/llm_provider.py`** – shared helper that returns a LangChain `ChatOpenAI` instance for any registered provider.
2. **Provider registry** – a dictionary mapping provider names to their configuration (base URL, API-key env var, default model).
3. **Conversational chain** – LangChain prompt + LLM + message history, identical regardless of which provider is active.

## Method Details

### Architecture
```
User Input
    │
    ▼
Prompt Template  ────▶  get_llm(provider)  ────▶  LLM Response
    ▲                       │
    │                  ┌────┴────┐
History Store          │ OpenAI  │
                       │ MiniMax │
                       │  ...    │
                       └─────────┘
```

The `get_llm()` function reads the provider configuration and returns the right `ChatOpenAI` object. The rest of the agent code never touches provider-specific details.

## Setup

### Install required packages

In [ ]:
# %pip install -q langchain langchain_openai openai python-dotenv

### Configure environment variables

Create a `.env` file in the repository root (or export the variables in your shell):

```bash
# Required for OpenAI provider
OPENAI_API_KEY=sk-...

# Required for MiniMax provider (get yours at https://www.minimaxi.com/)
MINIMAX_API_KEY=sk-...
```

### Import the multi-provider helper and LangChain components

In [ ]:
import sys, os

# Ensure the repository root is on the Python path so we can import utils/
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from utils.llm_provider import get_llm, list_providers
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

### List available providers

In [ ]:
print("Available LLM providers:", list_providers())

## Part 1 – Conversational Agent with OpenAI (baseline)

### Initialize the LLM via the unified helper

In [ ]:
llm_openai = get_llm(provider="openai", model="gpt-4o-mini", temperature=0)
print(f"Provider: OpenAI | Model: {llm_openai.model_name}")

### Build the conversational chain (provider-agnostic)

In [ ]:
def build_agent(llm):
    """Build a conversational agent chain for any LangChain-compatible LLM."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful AI assistant. Keep answers concise."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ])

    store: dict = {}

    def get_history(session_id: str):
        if session_id not in store:
            store[session_id] = ChatMessageHistory()
        return store[session_id]

    chain = prompt | llm
    return RunnableWithMessageHistory(
        chain,
        get_history,
        input_messages_key="input",
        history_messages_key="history",
    ), store

In [ ]:
agent_openai, history_openai = build_agent(llm_openai)

config = {"configurable": {"session_id": "demo_openai"}}

r1 = agent_openai.invoke({"input": "Hello! My name is Alice."}, config=config)
print("OpenAI:", r1.content)

r2 = agent_openai.invoke({"input": "What is my name?"}, config=config)
print("OpenAI:", r2.content)

## Part 2 – Switch to MiniMax M2.7

Switching providers is a one-line change.  The rest of the agent code stays exactly the same.

In [ ]:
llm_minimax = get_llm(provider="minimax")  # defaults to MiniMax-M2.7
print(f"Provider: MiniMax | Model: {llm_minimax.model_name}")

In [ ]:
agent_minimax, history_minimax = build_agent(llm_minimax)

config = {"configurable": {"session_id": "demo_minimax"}}

r1 = agent_minimax.invoke({"input": "Hello! My name is Alice."}, config=config)
print("MiniMax:", r1.content)

r2 = agent_minimax.invoke({"input": "What is my name?"}, config=config)
print("MiniMax:", r2.content)

### Use the high-speed variant for lower latency

In [ ]:
llm_fast = get_llm(provider="minimax", model="MiniMax-M2.5-highspeed")
agent_fast, _ = build_agent(llm_fast)

config = {"configurable": {"session_id": "demo_fast"}}
r = agent_fast.invoke({"input": "Explain quantum computing in two sentences."}, config=config)
print("MiniMax (highspeed):", r.content)

## Part 3 – Compare providers side-by-side

Run the same prompt through both providers and compare outputs.

In [ ]:
prompt_text = "What are the three most important factors when choosing a cloud LLM provider?"

for name, llm in [("OpenAI", llm_openai), ("MiniMax", llm_minimax)]:
    agent, _ = build_agent(llm)
    config = {"configurable": {"session_id": f"compare_{name}"}}
    resp = agent.invoke({"input": prompt_text}, config=config)
    print(f"\n--- {name} ---")
    print(resp.content)

## Part 4 – Environment-driven provider selection

In production, you often want the provider to be controlled by an environment variable rather than hard-coded.

In [ ]:
import os

# Set via environment (e.g. LLM_PROVIDER=minimax python app.py)
provider_name = os.getenv("LLM_PROVIDER", "minimax")
llm_env = get_llm(provider=provider_name)

agent_env, _ = build_agent(llm_env)
config = {"configurable": {"session_id": "env_demo"}}
r = agent_env.invoke({"input": "Summarize the benefits of multi-provider LLM setups."}, config=config)
print(f"[{provider_name}]:", r.content)

## Conclusion

By introducing a thin **provider abstraction** (`utils/llm_provider.py`), all tutorials in this repository can:

- Support **OpenAI** and **MiniMax** (and any future OpenAI-compatible provider) without code changes.
- Switch providers via a single parameter or environment variable.
- Share the same agent logic regardless of the underlying LLM.

### Next steps
- Explore more complex agents in this repository using the `get_llm()` helper.
- Add additional providers by extending `PROVIDERS` in `utils/llm_provider.py`.
- Try MiniMax-M2.5-highspeed for latency-sensitive workloads, or MiniMax-M2.7 for the latest capabilities.

### References
- [MiniMax Platform](https://www.minimaxi.com/)
- [MiniMax API Documentation](https://www.minimaxi.com/document/introduction)
- [LangChain OpenAI Integration](https://python.langchain.com/docs/integrations/chat/openai/)